# **RAG LLM Powered Food Recommendation & Query Response System**

## Install Dependencies

In [1]:
!pip install pandas sentence-transformers faiss-cpu huggingface-hub --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 56.1 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

## Data Prepration

In [3]:
menu = pd.DataFrame({
    'item_id': [1, 2, 3, 4, 5],
    'item_name': ['Paneer Tikka Pizza', 'Margherita Pizza', 'Chicken Tandoori Pizza', 'Veggie Paradise', 'Garlic Bread'],
    'description': ['Spicy paneer with capsicum & onions', 'Classic cheese & tomato pizza', 'Chicken tikka with onions & peppers', 'Veg pizza with babycorn & olives', 'Fresh garlic bread with herbs'],
    'category': ['Veg Pizza', 'Veg Pizza', 'Non-Veg Pizza', 'Veg Pizza', 'Sides'],
    'price': [320, 260, 380, 300, 150]
})

orders = pd.DataFrame({
    'order_id': [1001, 1002, 1003, 1004],
    'user_id': ['U1', 'U1', 'U1', 'U2'],
    'item_id': [1, 4, 1, 2],
    'rating': [5, 4, 5, 4],
    'ordered_at': ['2025-01-03 19:30', '2025-01-10 20:10', '2025-01-15 21:00', '2025-01-05 18:45']
})


In [4]:
menu.head()

,item_id,item_name,description,category,price
0,1,Paneer Tikka Pizza,Spicy paneer with capsicum & onions,Veg Pizza,320
1,2,Margherita Pizza,Classic cheese & tomato pizza,Veg Pizza,260
2,3,Chicken Tandoori Pizza,Chicken tikka with onions & peppers,Non-Veg Pizza,380
3,4,Veggie Paradise,Veg pizza with babycorn & olives,Veg Pizza,300
4,5,Garlic Bread,Fresh garlic bread with herbs,Sides,150


In [5]:
orders.head()

,order_id,user_id,item_id,rating,ordered_at
0,1001,U1,1,5,2025-01-03 19:30
1,1002,U1,4,4,2025-01-10 20:10
2,1003,U1,1,5,2025-01-15 21:00
3,1004,U2,2,4,2025-01-05 18:45


## Prepare text documents for Vector DB

In [6]:
# Create rich text representation for embeddings
menu["doc_text"] = menu["item_name"] + " " + menu["description"] + " " + menu["category"]

In [7]:
menu["doc_text"]

,doc_text
0,Paneer Tikka Pizza Spicy paneer with capsicum ...
1,Margherita Pizza Classic cheese & tomato pizza...
2,Chicken Tandoori Pizza Chicken tikka with onio...
3,Veggie Paradise Veg pizza with babycorn & oliv...
4,Garlic Bread Fresh garlic bread with herbs Sides


## Embeddings Setup

Here we are creating embedding with the help of Embedding model from Hugging face "**all-MiniLM-L6-v2"** it is light weight and fast.

In [8]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate Embeddings
item_embeddings = embed_model.encode(menu["doc_text"].tolist())
faiss.normalize_L2(item_embeddings) # Normalize for Cosine Similarity


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
item_embeddings

array([[-0.07978552,  0.01699249, -0.05825087, ...,  0.05939404,
        -0.03632474,  0.03966881],
       [-0.06536213,  0.0376624 , -0.04696827, ...,  0.02453765,
         0.05777392,  0.00226295],
       [-0.08732856,  0.05951275, -0.07354879, ...,  0.02357281,
         0.01061074,  0.01112188],
       [-0.06242159,  0.04121247, -0.04746523, ..., -0.00828142,
         0.03494671, -0.04430706],
       [-0.06739403,  0.06512342, -0.02641433, ...,  0.00625229,
        -0.06011075,  0.01023424]], dtype=float32)

In [28]:
print(f'Embeddings has a shape of : {item_embeddings.shape}')

Embeddings has a shape of : (5, 384)


## Build FAISS Vector Index

Here we are Creating a **Cosine-similarity** vector index (via inner product).

In [29]:
# Setup FAISS Index
dimension = item_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension) # Inner Product + Normalized = Cosine
index.add(item_embeddings)
print("Faiss vector index are built and reay to use")

System is ready to use


## Recommendation Logic

1. Understanding user history for User U1

2. Detecting Veg/Non-Veg preference

3. Building a user embedding vector

4. Searching for similar items via FAISS

5. Prioritizing items the user hasn't eaten

6. Falling back to similar items they have eaten if needed

In [11]:
def get_recommendations(user_id, top_n=3):
    # 1. Get History
    user_history = orders[orders["user_id"] == user_id]
    if user_history.empty:
        return menu.head(top_n)["item_name"].tolist()

    # 2. Define Filters
    liked_items = user_history[user_history["rating"] >= 4]["item_id"].unique()
    eaten_ids = user_history["item_id"].unique().tolist()

    # Check Veg Preference
    liked_cats = menu[menu["item_id"].isin(liked_items)]["category"].tolist()
    is_veg_fan = any("Veg" in c for c in liked_cats) and not any("Non-Veg" in c for c in liked_cats)

    # 3. Vector Search (User Profile)
    if len(liked_items) > 0:
        liked_indices = [menu[menu["item_id"] == i].index[0] for i in liked_items]
        user_vector = np.mean(item_embeddings[liked_indices], axis=0).reshape(1, -1)
    else:
        user_vector = item_embeddings # Fallback if no high ratings

    faiss.normalize_L2(user_vector)
    distances, indices = index.search(user_vector, k=len(menu))

    # 4. Fill Recommendations (Phase 1: Unseen Items)
    recommendations = []

    # Helper to check validity
    def is_valid(item_row):
        # Veg check
        if is_veg_fan and "Non-Veg" in item_row["category"]:
            return False
        return True

    # Pass 1: Add ONLY Unseen items
    for idx in indices[0]:
        item = menu.iloc[idx]
        if item["item_id"] not in eaten_ids and is_valid(item):
            recommendations.append(item["item_name"])

        if len(recommendations) >= top_n:
            break

    # Pass 2: Fallback (If we don't have 3 yet, add Repeat items)
    if len(recommendations) < top_n:
        for idx in indices[0]:
            item = menu.iloc[idx]
            # Add if it's NOT already in our list AND valid
            if item["item_name"] not in recommendations and is_valid(item):
                recommendations.append(item["item_name"])

            if len(recommendations) >= top_n:
                break

    return recommendations

In [12]:
# TEST Recommendations
print(f"\n📋 Recommendations for U1: \n{get_recommendations('U1')}")



📋 Recommendations for U1: 
['Margherita Pizza', 'Garlic Bread', 'Veggie Paradise']


## RAG ChatBot Function

**The rag_chatbot function works as follows**:

* A client for the Mistral 7B Instruct model is loaded from
Hugging Face.

* For each user query, it detects the intent of the query (whether it’s about order history or menu/recommendations).

* If it's a history question, it fetches the user’s last order from the order history.

* If it's a menu/info question, it performs a vector semantic search over the menu using embeddings and FAISS.

* It then injects the retrieved context (order history or menu items) into the prompt.

* Finally, it asks the LLM (Mistral) to generate a short, clear, conversational answer based on that context.

In [19]:
# Load LLM (Mistral)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except:
    hf_token = os.getenv("HF_TOKEN")

client = InferenceClient("mistralai/Mistral-7B-Instruct-v0.2", token=hf_token)

def rag_chatbot(query, user_id="U1"):
    # --- Step A: Intent Detection (Simple Keyword Check) ---
    query_lower = query.lower()
    is_history_query = any(w in query_lower for w in ["ordered", "last time", "history", "previous"])

    context = ""

    # --- Step B: Handle History Queries ---
    if is_history_query:
        user_hist = orders[orders["user_id"] == user_id].merge(menu, on="item_id")
        if not user_hist.empty:
            # Sort by date
            user_hist = user_hist.sort_values("ordered_at", ascending=False)
            latest = user_hist.iloc[0]
            context = f"User's Last Order: {latest['item_name']} on {latest['ordered_at']}. Rating: {latest['rating']}/5."
        else:
            context = "User has no order history."

    # --- Step C: Handle Menu/Recommendation Queries (Vector Search) ---
    else:
        q_emb = embed_model.encode([query])
        faiss.normalize_L2(q_emb)
        D, I = index.search(q_emb, k=3) # Get top 3 relevant items

        retrieved_items = menu.iloc[I[0]]
        context = "Menu Information:\n" + "\n".join(
            [f"- {row['item_name']} ({row['category']}): {row['description']} | Price: {row['price']}"
             for _, row in retrieved_items.iterrows()]
        )

    # --- Step D: Generate Answer with LLM ---
    system_prompt = f"""
    You are a restaurant assistant. Use the Context below to answer the user.

    CONTEXT:
    {context}

    RULES:
    1. NEVER start sentences with "Based on...", "Given your...", "Since you asked...", or "Taking into account..
    2. If the user asks about past orders, use the 'User's Last Order' info.
    3. Be short, punchy, and conversational.

    """

    response = client.chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        max_tokens=150,
        temperature=0.2
    )
    return response.choices[0].message.content


## ChatBot Demo

In [30]:
demo_queries = [
    "What is Paneer Tikka Pizza made of?",
    "Show me what I ordered last time",
    "Suggest something less spicy",
    "What did I eat last time?",
    "What should I eat today? I’m in the mood for something cheesy"
]

print("\n CHATBOT DEMO:")
for q in demo_queries:
    print(f"User: {q}")
    print(f"Bot : {rag_chatbot(q)}\n")


 CHATBOT DEMO:
User: What is Paneer Tikka Pizza made of?
Bot :  The Paneer Tikka Pizza is made with spicy paneer, capsicum, and onions.

User: Show me what I ordered last time
Bot :  Last time, you enjoyed a Paneer Tikka Pizza on January 15, 2025 at 9:00 PM. It received a perfect rating of 5/5!

User: Suggest something less spicy
Bot :  How about trying our Garlic Bread as a less spicy option? It's a delicious side with fresh garlic and herbs.

User: What did I eat last time?
Bot :  Last time, you enjoyed a Paneer Tikka Pizza on January 15, 2025 at 9:00 PM. It received a perfect 5-star rating!

User: What should I eat today? I’m in the mood for something cheesy
Bot :  Based on your current craving for something cheesy, I'd recommend the Margherita Pizza. It's a classic choice with a perfect balance of cheese and tomato. Enjoy your meal!

